# Validation SDP — K_CS wiring and ideal-switch blocker

This notebook validates the SDP wiring for `K_CS = C_AB + C_BA` on exact causally separable targets: white noise, fixed-order A≺B, fixed-order B≺A, and a convex mixture.

It deliberately does **not** pretend that the ideal quantum-switch benchmark is complete. The final cell verifies that `ideal_quantum_switch_process` still raises `NotImplementedError`. The repository must not be described as submission-ready until that benchmark is implemented and compared against a published reference.

In [ ]:
from pathlib import Path
import json

from deltawkrel.projectors import ProcessDims, assert_trace_convention, is_psd
from deltawkrel.switch_models import (
    white_noise_validation_process, fixed_order_A_before_B_process,
    fixed_order_B_before_A_process, causally_separable_mixture,
    ideal_quantum_switch_process,
)
from deltawkrel.sdp import solve_cs_robustness

ROOT = Path('..') if Path('../src').exists() else Path('.')
RESULTS = ROOT / 'results'
RESULTS.mkdir(exist_ok=True)
dims = ProcessDims(2,2,2,2)
print(dims)

## 1. Validate K_CS robustness on causally separable targets

For every target below, the calibrated robustness over `K_CS` should be numerically zero, because the target is already causally separable.

In [ ]:
targets = {
    'white_noise': white_noise_validation_process(dims),
    'fixed_order_A_before_B': fixed_order_A_before_B_process(dims),
    'fixed_order_B_before_A': fixed_order_B_before_A_process(dims),
    'causally_separable_mixture_q037': causally_separable_mixture(dims, q=0.37),
}
report = {'status': 'K_CS_infrastructure_validation', 'targets': {}}
for name, W in targets.items():
    assert_trace_convention(W, dims)
    assert is_psd(W), name
    diag = solve_cs_robustness(W, dims=dims, solver='CLARABEL')
    report['targets'][name] = diag.to_dict()
    print(name, diag)
    assert diag.status in {'optimal', 'optimal_inaccurate'}
    assert diag.objective_value < 1e-5

## 2. Ideal quantum switch benchmark: explicit blocker

The true ideal-switch validation requires the process matrix with its selected convention, including the future/global control system if used. This is the last scientific piece to implement before submission.

In [ ]:
try:
    ideal_quantum_switch_process(dims)
except NotImplementedError as exc:
    report['ideal_quantum_switch_benchmark'] = {
        'implemented': False,
        'submission_blocker': True,
        'message': str(exc),
    }
    print('Expected blocker:', exc)
else:
    raise AssertionError('The ideal switch must not silently return a placeholder.')

(RESULTS / 'sdp_validation_report.json').write_text(
    json.dumps(report, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8'
)
print('Report:', RESULTS / 'sdp_validation_report.json')

## Status

This notebook is a real SDP wiring validation for causally separable targets. It is **not yet** the published ideal quantum-switch robustness benchmark. Keep the repository marked as `not submission-ready` until the ideal-switch process and benchmark are added.